# 🧹 Data Cleaning
## Jumia Nigeria — Home & Kitchen Appliances

In this notebook, we take the **raw scraped data** from Phase 2 and transform it 
into a clean, analysis-ready dataset.

### What we'll do:
- Fix missing product names
- Clean price, discount, rating and review columns (remove ₦, %, commas etc.)
- Remove duplicate listings
- Create new calculated columns (discount amount, value score)
- Save the final cleaned dataset ready for analysis

**Input:** `data/raw/jumia_kitchen_raw.csv`  
**Output:** `data/cleaned/jumia_kitchen_cleaned.csv`

In [3]:
import pandas as pd
import numpy as np

# Load the raw scraped data
df = pd.read_csv("../data/raw/jumia_kitchen_raw.csv")

print(f"✅ Data loaded! Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("\n📋 First 5 rows:")
df.head()

✅ Data loaded! Shape: 215 rows × 8 columns

📋 First 5 rows:


,name,price,old_price,discount,rating,reviews,link,page
0,Hisense WSPA503 WM 5kg Twin Tub Machine,"₦ 170,000",NaN,32%,NaN,NaN,https://www.jumia.com.ng/hisense-wspa503-wm-5k...,1
1,3 Litres Whistling Gas & Stove Kettle,"₦ 12,000",NaN,NaN,NaN,NaN,https://www.jumia.com.ng/3-litres-whistling-ga...,1
2,Hisense 5kg Manual Washing Machine Twin Tub Wi...,"₦ 162,315",NaN,26%,NaN,NaN,https://www.jumia.com.ng/hisense-5kg-manual-wa...,1
3,Binatone 3 Litres Electric Jug (CEJ-3000) - Bl...,"₦ 31,919",NaN,36%,NaN,NaN,https://www.jumia.com.ng/binatone-3-litres-ele...,1
4,Legacy 8L Yampounder And Multifunctional Food ...,"₦ 46,500",NaN,15%,NaN,NaN,https://www.jumia.com.ng/8l-yampounder-and-mul...,1


In [6]:
print("📊 Data Info:")
print(df.info())

print("\n❓ Missing values per column:")
print(df.isnull().sum())

📊 Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   name       15 non-null     object
 1   price      215 non-null    object
 2   old_price  179 non-null    object
 3   discount   14 non-null     object
 4   rating     136 non-null    object
 5   reviews    136 non-null    object
 6   link       215 non-null    object
 7   page       215 non-null    int64 
dtypes: int64(1), object(7)
memory usage: 13.6+ KB
None

❓ Missing values per column:
name         200
price          0
old_price     36
discount     201
rating        79
reviews       79
link           0
page           0
dtype: int64


In [7]:
before = len(df)
df = df.drop_duplicates(subset=["name", "price"])
after = len(df)

print(f"🗑️  Removed {before - after} duplicate rows")
print(f"✅ Remaining rows: {after}")

🗑️  Removed 32 duplicate rows
✅ Remaining rows: 183


In [9]:
df["price_clean"] = (
    df["price"]
    .str.replace("₦", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["price_clean"] = pd.to_numeric(df["price_clean"], errors="coerce").round(2)

print("✅ Price column cleaned!")
print(df["price_clean"].describe().round(2))

✅ Price column cleaned!
count       177.00
mean      45369.88
std       60407.41
min        2185.00
25%       12000.00
50%       25999.00
75%       51000.00
max      468500.00
Name: price_clean, dtype: float64


In [10]:
df["old_price_clean"] = (
    df["old_price"]
    .str.replace("₦", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["old_price_clean"] = pd.to_numeric(df["old_price_clean"], errors="coerce")

print("✅ Old price column cleaned!")
print(f"Products with old price (on discount): {df['old_price_clean'].notna().sum()}")

✅ Old price column cleaned!
Products with old price (on discount): 145


In [14]:
df["discount_clean"] = (
    df["discount"]
    .str.replace("%", "", regex=False)
    .str.replace("-", "", regex=False)
    .str.strip()
)

df["discount_clean"] = pd.to_numeric(df["discount_clean"], errors="coerce")

print("✅ Discount column cleaned!")
print(df["discount_clean"].describe().round(2))

✅ Discount column cleaned!
count    14.00
mean     26.36
std      11.61
min       1.00
25%      17.50
50%      29.00
75%      35.00
max      43.00
Name: discount_clean, dtype: float64


In [15]:
# Since Jumia doesn't show ratings on listing pages,
# we fill missing values with 0 so they don't break our analysis

df["rating_clean"] = pd.to_numeric(df["rating"], errors="coerce").fillna(0)
df["reviews_clean"] = pd.to_numeric(df["reviews"], errors="coerce").fillna(0)

print("✅ Rating and reviews columns handled!")
print(f"Products with ratings: {(df['rating_clean'] > 0).sum()}")
print(f"Products without ratings: {(df['rating_clean'] == 0).sum()}")

✅ Rating and reviews columns handled!
Products with ratings: 0
Products without ratings: 183


In [16]:
# How much money you save
df["discount_amount"] = df["old_price_clean"] - df["price_clean"]

# Discount % calculated from prices
df["discount_pct_calc"] = (
    (df["discount_amount"] / df["old_price_clean"]) * 100
).round(2)

# Value score — higher discount + lower price = better deal
df["value_score"] = (
    df["discount_clean"] / df["price_clean"] * 10000
).round(4)

print("✅ New calculated columns created!")
print(df[["price_clean", "old_price_clean", "discount_amount", "discount_pct_calc", "value_score"]].head(10))

✅ New calculated columns created!
   price_clean  old_price_clean  discount_amount  discount_pct_calc  \
0     170000.0              NaN              NaN                NaN   
1      12000.0              NaN              NaN                NaN   
2     162315.0              NaN              NaN                NaN   
3      31919.0              NaN              NaN                NaN   
4      46500.0              NaN              NaN                NaN   
5      58900.0              NaN              NaN                NaN   
6      22894.0              NaN              NaN                NaN   
7      16339.0              NaN              NaN                NaN   
8     165000.0              NaN              NaN                NaN   
9      12500.0              NaN              NaN                NaN   

   value_score  
0       1.8824  
1          NaN  
2       1.6018  
3      11.2785  
4       3.2258  
5       0.1698  
6      18.7822  
7      22.0332  
8       1.5152  
9      11.2000

In [17]:
df_clean = df.drop(columns=["price", "old_price", "discount", "rating", "reviews"])

df_clean = df_clean.rename(columns={
    "price_clean":       "price",
    "old_price_clean":   "old_price",
    "discount_clean":    "discount_pct",
    "rating_clean":      "rating",
    "reviews_clean":     "reviews"
})

print("✅ Columns cleaned and renamed!")
print(df_clean.columns.tolist())

✅ Columns cleaned and renamed!
['name', 'link', 'page', 'price', 'old_price', 'discount_pct', 'rating', 'reviews', 'discount_amount', 'discount_pct_calc', 'value_score']


In [18]:
print("📊 Final cleaned data info:")
print(df_clean.info())

print(f"\n❓ Remaining missing values:")
print(df_clean.isnull().sum())

print(f"\n✅ Final shape: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")

df_clean.to_csv("../data/cleaned/jumia_kitchen_cleaned.csv", index=False)

print("\n💾 Saved to data/cleaned/jumia_kitchen_cleaned.csv")
print("🎉 Phase 3 Complete! Move on to 03_analysis.ipynb")

📊 Final cleaned data info:
<class 'pandas.core.frame.DataFrame'>
Index: 183 entries, 0 to 214
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   name               15 non-null     object 
 1   link               183 non-null    object 
 2   page               183 non-null    int64  
 3   price              177 non-null    float64
 4   old_price          145 non-null    float64
 5   discount_pct       14 non-null     float64
 6   rating             183 non-null    float64
 7   reviews            183 non-null    float64
 8   discount_amount    145 non-null    float64
 9   discount_pct_calc  145 non-null    float64
 10  value_score        14 non-null     float64
dtypes: float64(8), int64(1), object(2)
memory usage: 17.2+ KB
None

❓ Remaining missing values:
name                 168
link                   0
page                   0
price                  6
old_price             38
discount_pct         169
ra

In [22]:
# Check the cleaned dataframe properly
print("✅ Cleaned data sample:")
print(df_clean[["name", "price", "old_price", "discount_pct", "discount_amount", "discount_pct_calc"]].head(10))

print(f"\nProducts WITH old price: {df_clean['old_price'].notna().sum()}")
print(f"Products WITHOUT old price: {df_clean['old_price'].isna().sum()}")

print(f"\nProducts WITH discount: {df_clean['discount_pct'].notna().sum()}")
print(f"Products WITHOUT discount: {df_clean['discount_pct'].isna().sum()}")

✅ Cleaned data sample:
                                                name     price  old_price  \
0            Hisense WSPA503 WM 5kg Twin Tub Machine  170000.0        NaN   
1              3 Litres Whistling Gas & Stove Kettle   12000.0        NaN   
2  Hisense 5kg Manual Washing Machine Twin Tub Wi...  162315.0        NaN   
3  Binatone 3 Litres Electric Jug (CEJ-3000) - Bl...   31919.0        NaN   
4  Legacy 8L Yampounder And Multifunctional Food ...   46500.0        NaN   
5    Qasa 2 Litres (QBL-8008 Pro) Commercial Blender   58900.0        NaN   
6  Binatone 1.7 Litres (CEJ-1780) Electric Kettle...   22894.0        NaN   
7  Binatone Steam Iron (SI-1605) - Blue + 2 Years...   16339.0        NaN   
8  AKAI 4.0kg Washing Machine - Spinning + Draini...  165000.0        NaN   
9  ActiveO Bedbug And Insect Killer With Diatomac...   12500.0        NaN   

   discount_pct  discount_amount  discount_pct_calc  
0          32.0              NaN                NaN  
1           NaN      

In [24]:
import pandas as pd

# Reload the RAW data fresh
df_raw = pd.read_csv("../data/raw/jumia_kitchen_raw.csv")

print("Raw data shape:", df_raw.shape)
print("\nSample names from RAW file:")
print(df_raw["name"].head(10).tolist())
print(f"\nNaN names in RAW file: {df_raw['name'].isna().sum()}")

Raw data shape: (215, 8)

Sample names from RAW file:
['Hisense WSPA503 WM 5kg Twin Tub Machine', '3 Litres Whistling Gas & Stove Kettle', 'Hisense 5kg Manual Washing Machine Twin Tub With Lint Filter', 'Binatone 3 Litres Electric Jug (CEJ-3000) - Black + 2 Years Warranty', 'Legacy 8L Yampounder And Multifunctional Food Machine Cooper Motor', 'Qasa 2 Litres (QBL-8008 Pro) Commercial Blender', 'Binatone 1.7 Litres (CEJ-1780) Electric Kettle - White + 2 Years Warranty', 'Binatone Steam Iron (SI-1605) - Blue + 2 Years Warranty', 'AKAI 4.0kg Washing Machine - Spinning + Draining Function', 'ActiveO Bedbug And Insect Killer With Diatomaceous Earth - 500g']

NaN names in RAW file: 200


In [25]:
import pandas as pd

df_raw = pd.read_csv("../data/raw/jumia_kitchen_raw.csv")

# Extract product name from URL
# Example: /binatone-3-litres-electric-jug-71234.html → Binatone 3 Litres Electric Jug
def name_from_url(url):
    if not isinstance(url, str):
        return "Unknown"
    # Get the part between last / and .html
    slug = url.split("/")[-1].replace(".html", "")
    # Remove the product ID number at the end (e.g. -71786425)
    slug = "-".join(slug.split("-")[:-1])
    # Convert dashes to spaces and title case
    return slug.replace("-", " ").title()

df_raw["name"] = df_raw["name"].fillna(df_raw["link"].apply(name_from_url))

print(f"✅ Names recovered!")
print(f"Remaining NaN names: {df_raw['name'].isna().sum()}")
print(f"\nSample names:")
print(df_raw["name"].head(15).tolist())

✅ Names recovered!
Remaining NaN names: 0

Sample names:
['Hisense WSPA503 WM 5kg Twin Tub Machine', '3 Litres Whistling Gas & Stove Kettle', 'Hisense 5kg Manual Washing Machine Twin Tub With Lint Filter', 'Binatone 3 Litres Electric Jug (CEJ-3000) - Black + 2 Years Warranty', 'Legacy 8L Yampounder And Multifunctional Food Machine Cooper Motor', 'Qasa 2 Litres (QBL-8008 Pro) Commercial Blender', 'Binatone 1.7 Litres (CEJ-1780) Electric Kettle - White + 2 Years Warranty', 'Binatone Steam Iron (SI-1605) - Blue + 2 Years Warranty', 'AKAI 4.0kg Washing Machine - Spinning + Draining Function', 'ActiveO Bedbug And Insect Killer With Diatomaceous Earth - 500g', 'SILVER CREST German Industrial Food Crusher Blender, EXTRA MILL JAR', 'Century Quality Electric Spray & Steam Iron.', 'Blender + Electric Cooker + Electric Kettle + Toaster', 'SILVER CREST German Industrial 8500W Food Crusher Blender, EXTRA MILL JAR', 'SILVER CREST 2 Litres German Industrial Blender 8500W']


In [26]:
import pandas as pd

# Start fresh from raw data with all names recovered
df = df_raw.copy()

# --- Clean price ---
df["price"] = (
    df["price"]
    .str.replace("₦", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
df["price"] = pd.to_numeric(df["price"], errors="coerce").round(2)

# --- Clean old price ---
df["old_price"] = (
    df["old_price"]
    .str.replace("₦", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
df["old_price"] = pd.to_numeric(df["old_price"], errors="coerce").round(2)

# --- Clean discount ---
df["discount_pct"] = (
    df["discount"]
    .str.replace("%", "", regex=False)
    .str.replace("-", "", regex=False)
    .str.strip()
)
df["discount_pct"] = pd.to_numeric(df["discount_pct"], errors="coerce").round(2)

# --- Drop original discount column ---
df = df.drop(columns=["discount"])

# --- Remove duplicates ---
before = len(df)
df = df.drop_duplicates(subset=["name", "price"])
after = len(df)
print(f"🗑️ Removed {before - after} duplicates. Remaining: {after}")

# --- Calculated columns ---
df["discount_amount"] = (df["old_price"] - df["price"]).round(2)
df["discount_pct_calc"] = ((df["discount_amount"] / df["old_price"]) * 100).round(2)
df["value_score"] = (df["discount_pct"] / df["price"] * 10000).round(4)

# --- Extract brand ---
not_brands = [
    "3", "2", "1", "4", "5", "6", "7", "8", "9", "10",
    "Blender", "Generic", "New", "Mini", "Large", "Small",
    "Litres", "Litre", "Liter", "Liters", "KG", "Kg", "kg",
    "Electric", "Digital", "Automatic", "Manual", "Multi",
    "Hot", "Cold", "Double", "Single", "Triple", "High",
    "Low", "Ultra", "Super", "Pro", "Plus", "Max", "Air",
    "+", "-", "&", "HD", "LED", "LCD"
]

def extract_brand(name):
    if not isinstance(name, str):
        return "Unknown"
    words = name.split()
    for word in words:
        if word not in not_brands:
            return word
    return "Unknown"

df["brand"] = df["name"].apply(extract_brand)

# --- Final check ---
print(f"\n✅ Final shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\n🏭 Top 15 Brands:")
print(df["brand"].value_counts().head(15))

# --- Save ---
df.to_csv("../data/cleaned/jumia_kitchen_cleaned.csv", index=False)
df.to_excel("../data/cleaned/jumia_final.xlsx", index=False)

print("\n💾 Both files saved!")

🗑️ Removed 5 duplicates. Remaining: 210

✅ Final shape: 210 rows × 12 columns

🏭 Top 15 Brands:
brand
Nine            16
Binatone         9
Deadbug          8
Stainless        7
Rechargeable     6
Silver           6
Sonik            6
Syinix           5
Iron             5
Qasa             5
Legacy           5
Yam              5
King             4
Rico             4
Whistling        4
Name: count, dtype: int64

💾 Both files saved!


In [27]:
not_brands = [
    # Numbers
    "3", "2", "1", "4", "5", "6", "7", "8", "9", "10", "12", "15", "20",
    # Sizes and units
    "Litres", "Litre", "Liter", "Liters", "KG", "Kg", "kg", "L",
    "500g", "500ml", "1000w", "2000w",
    # Descriptive words
    "Blender", "Generic", "New", "Mini", "Large", "Small", "Heavy",
    "Electric", "Digital", "Automatic", "Manual", "Multi", "Duty",
    "Hot", "Cold", "Double", "Single", "Triple", "High", "Quality",
    "Low", "Ultra", "Super", "Pro", "Plus", "Max", "Air", "Best",
    "Stainless", "Steel", "Iron", "Rechargeable", "Portable",
    "Whistling", "Gas", "Commercial", "Industrial", "German",
    "Yam", "Deadbug", "Nine", "King", "Insect", "Bed", "Bug",
    "Spray", "Steam", "Wash", "Food", "Home", "Kitchen",
    # Symbols
    "+", "-", "&", "HD", "LED", "LCD", "W", "V"
]

def extract_brand(name):
    if not isinstance(name, str):
        return "Unknown"
    words = name.split()
    for word in words:
        if word not in not_brands:
            return word
    return "Unknown"

df["brand"] = df["name"].apply(extract_brand)

print("✅ Brands updated!")
print(df["brand"].value_counts().head(20))

# Save
df.to_csv("../data/cleaned/jumia_kitchen_cleaned.csv", index=False)
df.to_excel("../data/cleaned/jumia_final.xlsx", index=False)
print("\n💾 Files saved!")

✅ Brands updated!
brand
Binatone           9
Pounder            8
Organic            7
Kettle             6
Silver             6
Sonik              6
Legacy             5
Qasa               5
Syinix             5
Rico               4
Usb                4
Style              4
1.5L               4
Mosquito           4
Multifunctional    3
3L                 3
SILVER             3
In                 3
3Pcsset            3
Processor          3
Name: count, dtype: int64

💾 Files saved!


In [28]:
# Known real brands on Jumia Nigeria Home & Kitchen
real_brands = [
    "Binatone", "Hisense", "SILVER", "Silver", "Sonik", "Syinix",
    "Qasa", "Legacy", "Rico", "AKAI", "Akai", "ActiveO", "Century",
    "Nexus", "Thermocool", "Scanfrost", "LG", "Samsung", "Panasonic",
    "Philips", "Midea", "Haier", "Polystar", "Bruhm", "Maxi",
    "Nunix", "Abans", "Kenwood", "Moulinex", "Tefal", "Rebune",
    "Sayona", "Ramtons", "Blueflame", "Nasco", "Amaze", "Skyrun",
    "Tokunbo", "Maxim", "Westpool", "Sona", "Ignis", "Ariston"
]

def extract_brand(name):
    if not isinstance(name, str):
        return "Other"
    words = name.split()
    # First check if any word matches a known brand
    for word in words:
        if word in real_brands:
            return word
    # If no known brand found, return first word as fallback
    return "Other"

df["brand"] = df["name"].apply(extract_brand)

print("✅ Brands updated!")
print(df["brand"].value_counts().head(20))
print(f"\nOther (unrecognized brands): {(df['brand'] == 'Other').sum()}")

# Save
df.to_csv("../data/cleaned/jumia_kitchen_cleaned.csv", index=False)
df.to_excel("../data/cleaned/jumia_final.xlsx", index=False)
print("\n💾 Files saved!")

✅ Brands updated!
brand
Other        151
Binatone      11
Sonik          7
Silver         7
Legacy         6
Syinix         6
Qasa           5
Century        4
Rico           4
SILVER         3
Hisense        2
AKAI           1
ActiveO        1
Akai           1
Panasonic      1
Name: count, dtype: int64

Other (unrecognized brands): 151

💾 Files saved!


In [29]:
# Show us what's hiding in "Other"
other_products = df[df["brand"] == "Other"]["name"].head(30).tolist()
for name in other_products:
    print(name)

3 Litres Whistling Gas & Stove Kettle
Blender + Electric Cooker + Electric Kettle + Toaster
Electric Iron Pressing Clothes 1000W Hansen
Tropicwhirl Dry Iron 1000W White Blue
Boscon 1.8 Liters Powerful Fast Heating Electric Jug Kettle
Ridax Bedbugs Killer Spray
8 In 1 Multi Purpose Grinder 4.0L 9500W Bardefu
Generic 3 In 1 Mini Dry Portable Handheld Usb Rechargeable Wireless Car Vacuum Cleaner
Taigexin Hanging Electric Mosquito Insect Killer Range Lamp 6W
Nine 1.8L Electric Kettle Sh 01
Generic Electric Iron Multifunctional Dry Ironing Black
Generic Double Sided Glass Cleaning Tool For Windows Mirrors And Shower Doors
Generic Solar Inverter Low Voltage Generator Friendly Pressing Iron
Orvica Ultraglide Steam Iron Dry Iron Energy Efficient Pro Steam Technology Infused Ceramic Soleplate Nonstick Variable Temperature And Self Clean Function.
Generic Electric Cup Mini Electric Cooking Pot All In One
Timbutus Wireless Vacuum Cleaner Rechargeable For Car Home 120W
Generic Lm No More Sweat Pre

In [30]:
def extract_brand(name):
    if not isinstance(name, str):
        return "Other"
    
    # Known real brands whitelist
    real_brands = [
        "Binatone", "Hisense", "Sonik", "Syinix", "Qasa", "Legacy",
        "Rico", "SILVER", "Silver", "AKAI", "Akai", "ActiveO", "Century",
        "Nexus", "Thermocool", "Scanfrost", "LG", "Samsung", "Panasonic",
        "Philips", "Midea", "Haier", "Polystar", "Bruhm", "Maxi",
        "Nunix", "Kenwood", "Moulinex", "Tefal", "Rebune", "Boscon",
        "Tropicwhirl", "Ridax", "Bardefu", "Taigexin", "Nine", "Orvica",
        "Timbutus", "Mythco", "Teknocool", "Addigoes", "Hansen",
        "Sayona", "Ramtons", "Skyrun", "Sona", "Ariston"
    ]
    
    # Check for Generic first
    if name.startswith("Generic"):
        return "Generic"
    
    words = name.split()
    
    # Check whitelist
    for word in words:
        if word in real_brands:
            return word
    
    # If no match, first word is likely the brand
    return words[0] if words else "Other"

df["brand"] = df["name"].apply(extract_brand)

# Merge SILVER and Silver into one
df["brand"] = df["brand"].replace("Silver", "SILVER")

# Merge AKAI and Akai into one  
df["brand"] = df["brand"].replace("Akai", "AKAI")

print("✅ Brands finalised!")
print(df["brand"].value_counts().head(20))
print(f"\nRemaining Other: {(df['brand'] == 'Other').sum()}")

# Save
df.to_csv("../data/cleaned/jumia_kitchen_cleaned.csv", index=False)
df.to_excel("../data/cleaned/jumia_final.xlsx", index=False)
print("\n💾 Files saved!")

✅ Brands finalised!
brand
Generic     81
Nine        16
Binatone    11
SILVER       9
Deadbug      7
Sonik        7
Legacy       6
Syinix       6
Qasa         5
Rico         4
King         4
Century      3
Sonifer      2
3            2
Taigexin     2
Addigoes     2
Timbutus     2
Hisense      2
Electric     2
AKAI         2
Name: count, dtype: int64

Remaining Other: 0

💾 Files saved!


In [31]:
# Fix remaining bad brand names
df["brand"] = df["brand"].replace({
    "3": "Generic",
    "Electric": "Generic", 
    "Deadbug": "Generic",
    "King": "Generic"
})

print("✅ Final brand cleanup done!")
print(df["brand"].value_counts().head(20))

# Save final files
df.to_csv("../data/cleaned/jumia_kitchen_cleaned.csv", index=False)
df.to_excel("../data/cleaned/jumia_final.xlsx", index=False)
print("\n💾 Files saved! ✅")

✅ Final brand cleanup done!
brand
Generic     96
Nine        16
Binatone    11
SILVER       9
Sonik        7
Legacy       6
Syinix       6
Qasa         5
Rico         4
Century      3
Timbutus     2
Addigoes     2
Taigexin     2
Hisense      2
AKAI         2
Sonifer      2
Cast         1
Bedbug       1
Genera       1
Round        1
Name: count, dtype: int64

💾 Files saved! ✅


In [32]:
df["brand"] = df["brand"].replace({
    "Cast": "Generic",
    "Bedbug": "Generic",
    "Genera": "Generic",
    "Round": "Generic"
})

print("✅ Brands are clean!")
print(df["brand"].value_counts().head(20))

# Save final files
df.to_csv("../data/cleaned/jumia_kitchen_cleaned.csv", index=False)
df.to_excel("../data/cleaned/jumia_final.xlsx", index=False)
print("\n💾 Files saved! ✅")

✅ Brands are clean!
brand
Generic     100
Nine         16
Binatone     11
SILVER        9
Sonik         7
Legacy        6
Syinix        6
Qasa          5
Rico          4
Century       3
Hisense       2
Sonifer       2
Timbutus      2
Taigexin      2
Addigoes      2
AKAI          2
Ridax         1
Bardefu       1
Foldable      1
3L            1
Name: count, dtype: int64

💾 Files saved! ✅


In [33]:
df["brand"] = df["brand"].replace({
    "Foldable": "Generic",
    "3L": "Generic"
})

print("✅ Final brand list:")
print(df["brand"].value_counts().head(20))

# Save final files
df.to_csv("../data/cleaned/jumia_kitchen_cleaned.csv", index=False)
df.to_excel("../data/cleaned/jumia_final.xlsx", index=False)
print("\n💾 Files saved! ✅")

✅ Final brand list:
brand
Generic      102
Nine          16
Binatone      11
SILVER         9
Sonik          7
Legacy         6
Syinix         6
Qasa           5
Rico           4
Century        3
Addigoes       2
Timbutus       2
Sonifer        2
Taigexin       2
Hisense        2
AKAI           2
Firman         1
World          1
Panasonic      1
Pizza          1
Name: count, dtype: int64

💾 Files saved! ✅


In [34]:
df["brand"] = df["brand"].replace({
    "World": "Generic",
    "Pizza": "Generic"
})

print("✅ Final brand list:")
print(df["brand"].value_counts().head(20))

# Save final files
df.to_csv("../data/cleaned/jumia_kitchen_cleaned.csv", index=False)
df.to_excel("../data/cleaned/jumia_final.xlsx", index=False)
print("\n💾 Files saved! ✅")

✅ Final brand list:
brand
Generic      104
Nine          16
Binatone      11
SILVER         9
Sonik          7
Legacy         6
Syinix         6
Qasa           5
Rico           4
Century        3
Addigoes       2
Timbutus       2
Taigexin       2
Sonifer        2
Hisense        2
AKAI           2
Two            1
Firman         1
Panasonic      1
Japan          1
Name: count, dtype: int64

💾 Files saved! ✅


In [35]:
df["brand"] = df["brand"].replace({
    "Two": "Generic",
    "Japan": "Generic"
})

print("✅ Final brand list:")
print(df["brand"].value_counts().head(20))

# Save final files
df.to_csv("../data/cleaned/jumia_kitchen_cleaned.csv", index=False)
df.to_excel("../data/cleaned/jumia_final.xlsx", index=False)
print("\n💾 Files saved! ✅")

✅ Final brand list:
brand
Generic       106
Nine           16
Binatone       11
SILVER          9
Sonik           7
Legacy          6
Syinix          6
Qasa            5
Rico            4
Century         3
Hisense         2
Sonifer         2
Addigoes        2
Taigexin        2
Timbutus        2
AKAI            2
Ridax           1
Bardefu         1
Citel           1
Emouliness      1
Name: count, dtype: int64

💾 Files saved! ✅


In [36]:
# Save with the exact same sheet name Power BI expects
df.to_excel("../data/cleaned/jumia_final.xlsx", 
            sheet_name="jumia_kitchen_cleaned",  # ← same name as before
            index=False)

print("💾 Saved with correct sheet name!")

💾 Saved with correct sheet name!


In [37]:
df["brand"] = df["brand"].replace({
    "Yam": "Generic",
    "Tower": "Generic",
    "1000W": "Generic",
    "Dsp": "Generic",
    "Tropicwhirl": "Generic"
})

print("✅ Final cleanup done!")
print(df["brand"].value_counts().head(20))

df.to_excel("../data/cleaned/jumia_final.xlsx",
            sheet_name="jumia_kitchen_cleaned",
            index=False)
print("💾 Saved!")

✅ Final cleanup done!
brand
Generic      111
Nine          16
Binatone      11
SILVER         9
Sonik          7
Legacy         6
Syinix         6
Qasa           5
Rico           4
Century        3
Sonifer        2
Addigoes       2
Timbutus       2
Taigexin       2
Hisense        2
AKAI           2
Mr.            1
Gourmet        1
Firman         1
Panasonic      1
Name: count, dtype: int64
💾 Saved!


In [38]:
df["brand"] = df["brand"].replace({
    "Organic": "Generic",
    "Blender": "Generic",
    "Bardefu": "Generic"
})

print(df["brand"].value_counts().head(20))

df.to_excel("../data/cleaned/jumia_final.xlsx",
            sheet_name="jumia_kitchen_cleaned",
            index=False)
print("💾 Saved!")

brand
Generic     114
Nine         16
Binatone     11
SILVER        9
Sonik         7
Syinix        6
Legacy        6
Qasa          5
Rico          4
Century       3
Taigexin      2
Hisense       2
Addigoes      2
Sonifer       2
AKAI          2
Timbutus      2
Mr.           1
Nexo          1
Firman        1
Fast          1
Name: count, dtype: int64
💾 Saved!


In [39]:
print("Let's see ALL Binatone products:")
print(df[df["name"].str.contains("Binatone", case=False, na=False)][["name", "brand"]])

Let's see ALL Binatone products:
                                                  name     brand
3    Binatone 3 Litres Electric Jug (CEJ-3000) - Bl...  Binatone
6    Binatone 1.7 Litres (CEJ-1780) Electric Kettle...  Binatone
7    Binatone Steam Iron (SI-1605) - Blue + 2 Years...  Binatone
18           Binatone Dry Iron Di1255 2 Years Warranty  Binatone
28   Binatone 1.5 Litres Blg 403 Blender Black 2 Ye...  Binatone
30          Smooth Gliding Steam Iron Si 2410 Binatone  Binatone
38   1.7 Litres Auto Stop Electric Kettle Cej 1750 ...  Binatone
85   Binatone 1.5 Litres Blg 620 Blendergrinder Wit...  Binatone
165   Binatone 6 Litres All In One Yam Pounder Kc 6000  Binatone
181   Binatone 6 Litres All In One Yam Pounder Kc 6000  Binatone
198   Binatone 6 Litres All In One Yam Pounder Kc 6000  Binatone


In [40]:
print("Top 15 brands excluding Generic:")
print(df[df["brand"] != "Generic"]["brand"].value_counts().head(15))

Top 15 brands excluding Generic:
brand
Nine        16
Binatone    11
SILVER       9
Sonik        7
Legacy       6
Syinix       6
Qasa         5
Rico         4
Century      3
Hisense      2
Taigexin     2
Sonifer      2
Addigoes     2
Timbutus     2
AKAI         2
Name: count, dtype: int64
